### Procesamiento de Lenguaje Natural I
# **Desafío 1 — Resolución**

Vectorización, similaridad, clasificación por prototipos y Naïve Bayes sobre **20 Newsgroups**.

In [ ]:
# Instalación de dependencias compatible con Colab, venv y entornos PEP 668 (Debian/Ubuntu).
# Si las librerías ya están instaladas, esta celda no hace nada.
import importlib.util, subprocess, sys

_paquetes = {'numpy': 'numpy', 'sklearn': 'scikit-learn'}
_faltantes = [pip_name for mod, pip_name in _paquetes.items()
              if importlib.util.find_spec(mod) is None]

if _faltantes:
    print(f'Instalando: {_faltantes}')
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *_faltantes]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        # Fallback para entornos PEP 668 (externally-managed)
        cmd.append('--break-system-packages')
        subprocess.run(cmd, check=True)
    print('Listo.')
else:
    print('numpy y scikit-learn ya están instalados.')

In [2]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## Carga de datos

Quitamos `headers`, `footers` y `quotes` para que el modelo no aprenda atajos triviales (firmas, citas) y se enfoque en el contenido.

In [3]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test  = fetch_20newsgroups(subset='test',  remove=('headers', 'footers', 'quotes'))

y_train = newsgroups_train.target
y_test  = newsgroups_test.target
target_names = newsgroups_train.target_names

print(f'Train: {len(newsgroups_train.data)} docs | Test: {len(newsgroups_test.data)} docs | Clases: {len(target_names)}')

Train: 11314 docs | Test: 7532 docs | Clases: 20


## Vectorización base (TF-IDF)

Usamos esta vectorización como punto de partida común para las consignas 1, 2 y 4.

In [4]:
tfidfvect = TfidfVectorizer()
X_train = tfidfvect.fit_transform(newsgroups_train.data)
X_test  = tfidfvect.transform(newsgroups_test.data)

print(f'X_train shape: {X_train.shape}  |  X_test shape: {X_test.shape}')

X_train shape: (11314, 101631)  |  X_test shape: (7532, 101631)


---
## **1. Similaridad entre documentos**

Elegimos 5 documentos al azar del set de entrenamiento y para cada uno mostramos los 5 más similares según similaridad coseno con su vector TF-IDF.

In [5]:
random_idx = rng.choice(X_train.shape[0], size=5, replace=False)
print('Documentos elegidos al azar:', random_idx)

for idx in random_idx:
    sims = cosine_similarity(X_train[idx], X_train)[0]
    # excluimos el propio documento (que tendría similaridad 1)
    top5 = np.argsort(sims)[::-1][1:6]

    print('=' * 90)
    print(f'DOC {idx}  |  clase real: {target_names[y_train[idx]]}')
    print('-' * 90)
    snippet = newsgroups_train.data[idx].strip().replace('\n', ' ')[:300]
    print(f'Texto: {snippet}...')
    print('-' * 90)
    print('Top 5 documentos más similares:')
    for j in top5:
        snip_j = newsgroups_train.data[j].strip().replace('\n', ' ')[:120]
        print(f'  idx={j:5d}  sim={sims[j]:.3f}  clase={target_names[y_train[j]]:30s} | {snip_j}...')
    print()

Documentos elegidos al azar: [8754 4965 7404 1009 4899]
DOC 8754  |  clase real: talk.religion.misc
------------------------------------------------------------------------------------------
Texto: /(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves.  They may be permanently damaging their body.  That is true.  It is also none of your business.    Some people may also reason that by reading the bible and being a Xtian you are permanentl...
------------------------------------------------------------------------------------------
Top 5 documentos más similares:
  idx= 6552  sim=0.490  clase=talk.religion.misc             | If I have a habit that I really want to break, and I am willing to make whatever sacrifice I need to make to break it, t...
  idx=10613  sim=0.481  clase=talk.religion.misc             | /(hudson) /Yes you do.  Who is to say that it is immoral for onesself to experience /pain or to be hurt in some other wa...
 

### Interpretación (consigna 1)

Mirando los 5 documentos elegidos al azar (indices 8754, 4965, 7404, 1009, 4899):

- **Doc 8754** (`talk.religion.misc`): 4 de los 5 vecinos son de la misma clase. El único "intruso" es `talk.politics.mideast`, una clase temáticamente vecina (debates sobre religión y política suelen compartir vocabulario).
- **Doc 4965** (`comp.sys.mac.hardware`): 2 vecinos de la misma clase, 2 de `comp.sys.ibm.pc.hardware` y 1 de `comp.graphics`. **Todos del cluster `comp.*`** — tiene sentido total, son temas de computación con vocabulario muy compartido (printer, port, modem, etc.).
- **Doc 7404** (`comp.os.ms-windows.misc`): los vecinos son `comp.windows.x` y `comp.graphics`. Cero match exacto pero **todos comp.\***. El doc consulta sobre minimizar program manager (windows): nada raro que se parezca a posts sobre X-Window.
- **Doc 1009** (`talk.politics.guns`): 4 de 5 mismos. El intruso es `alt.atheism`, otro grupo de debate.
- **Doc 4899** (`sci.crypt`): 3 de 5 mismos, los otros son grupos de debate político/ideológico — el texto cita a Chomsky y habla de control de información, así que la similaridad léxica con esos foros es razonable.

**Conclusiones generales:**
- TF-IDF + coseno captura bien la temática: ~70-80% de los vecinos son de la misma clase.
- Los "errores" caen casi siempre en **clases temáticamente cercanas** (los 6 clusters de 20 Newsgroups: comp, rec, sci, talk, religion, misc) — no son errores semánticos sino reflejos de la similitud real entre temas.
- Los valores absolutos de similaridad son bajos (0.13–0.49) porque los vectores son muy esparsos en un vocab de 101k términos; lo que importa es el **ranking relativo**, no el valor absoluto.

---
## **2. Clasificador por prototipos (1-NN coseno, tipo zero-shot)**

Asignamos a cada documento de test la etiqueta del documento de train con mayor similaridad coseno.

Cálculo eficiente: con vectores TF-IDF ya normalizados a norma L2, el producto `X_test @ X_train.T` **es** la matriz de similaridades coseno. Evita iterar documento por documento.

In [6]:
# X_train y X_test ya están L2-normalizados por TfidfVectorizer (norm='l2' por defecto)
sim_test_train = X_test @ X_train.T          # sparse @ sparse.T -> resultado denso pequeño por filas
nearest = np.asarray(sim_test_train.argmax(axis=1)).ravel()
y_pred_proto = y_train[nearest]

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f'F1-macro del clasificador por prototipos (1-NN coseno): {f1_proto:.4f}')

F1-macro del clasificador por prototipos (1-NN coseno): 0.5050


### Interpretación (consigna 2)

**Resultado: F1-macro = 0.5050**

Es un baseline razonable considerando que el modelo no aprende nada: solo memoriza el train y compara documento por documento. Las limitaciones son claras:

- Decide en base a **un único vecino**, así que es muy sensible a documentos atípicos o ruidosos del train.
- No agrega evidencia entre múltiples documentos de la misma clase.
- No pondera la confianza de la predicción (a diferencia de Naïve Bayes que devuelve probabilidades).

Como veremos en la consigna 3, Naïve Bayes va a saltar este baseline en ~+0.19 puntos de F1-macro.

---
## **3. Maximizar F1-macro con Naïve Bayes**

Probamos combinaciones de:
- Vectorizador: `CountVectorizer` vs `TfidfVectorizer`.
- Hiperparámetros: `min_df`, `max_df`, `stop_words`, `sublinear_tf` (sólo TF-IDF).
- Modelo: `MultinomialNB` vs `ComplementNB`, variando `alpha`.

**Restricción:** no tocamos `ngram_range` (queda en `(1,1)` por defecto).

In [7]:
def evaluar(vectorizer, model, nombre):
    Xtr = vectorizer.fit_transform(newsgroups_train.data)
    Xte = vectorizer.transform(newsgroups_test.data)
    model.fit(Xtr, y_train)
    yp = model.predict(Xte)
    f1 = f1_score(y_test, yp, average='macro')
    print(f'{nombre:65s}  F1-macro = {f1:.4f}')
    return f1

resultados = {}

# --- Baselines: configuraciones por defecto ---
resultados['Count + MultinomialNB (default)'] = evaluar(
    CountVectorizer(), MultinomialNB(), 'Count default + MultinomialNB')
resultados['Tfidf + MultinomialNB (default)'] = evaluar(
    TfidfVectorizer(), MultinomialNB(), 'Tfidf default + MultinomialNB')
resultados['Tfidf + ComplementNB (default)'] = evaluar(
    TfidfVectorizer(), ComplementNB(), 'Tfidf default + ComplementNB')

# --- Con stop_words='english' ---
resultados['Tfidf(stop=en) + ComplementNB'] = evaluar(
    TfidfVectorizer(stop_words='english'), ComplementNB(),
    'Tfidf stop_words=english + ComplementNB')

# --- Filtros de frecuencia + sublinear_tf ---
resultados['Tfidf(min_df=2, max_df=0.95, sublinear) + ComplementNB'] = evaluar(
    TfidfVectorizer(stop_words='english', min_df=2, max_df=0.95, sublinear_tf=True),
    ComplementNB(),
    'Tfidf min_df=2 max_df=0.95 sublinear + ComplementNB')

# --- Mismo vectorizador, MultinomialNB para comparar ---
resultados['Tfidf(min_df=2, max_df=0.95, sublinear) + MultinomialNB'] = evaluar(
    TfidfVectorizer(stop_words='english', min_df=2, max_df=0.95, sublinear_tf=True),
    MultinomialNB(),
    'Tfidf min_df=2 max_df=0.95 sublinear + MultinomialNB')

# --- Búsqueda fina sobre alpha de ComplementNB con la mejor vectorización ---
print('\nBúsqueda de alpha para ComplementNB con el mejor vectorizador:')
best_f1, best_alpha = 0.0, None
vect_best = TfidfVectorizer(stop_words='english', min_df=2, max_df=0.95, sublinear_tf=True)
Xtr = vect_best.fit_transform(newsgroups_train.data)
Xte = vect_best.transform(newsgroups_test.data)
for alpha in [0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0]:
    clf = ComplementNB(alpha=alpha).fit(Xtr, y_train)
    f1 = f1_score(y_test, clf.predict(Xte), average='macro')
    print(f'  alpha={alpha:<5} -> F1-macro = {f1:.4f}')
    if f1 > best_f1:
        best_f1, best_alpha = f1, alpha

print(f'\n>>> Mejor configuración: ComplementNB(alpha={best_alpha}) con Tfidf afinado -> F1-macro = {best_f1:.4f}')

Count default + MultinomialNB                                      F1-macro = 0.5121


Tfidf default + MultinomialNB                                      F1-macro = 0.5854


Tfidf default + ComplementNB                                       F1-macro = 0.6930


Tfidf stop_words=english + ComplementNB                            F1-macro = 0.6936


Tfidf min_df=2 max_df=0.95 sublinear + ComplementNB                F1-macro = 0.6921


Tfidf min_df=2 max_df=0.95 sublinear + MultinomialNB               F1-macro = 0.6437

Búsqueda de alpha para ComplementNB con el mejor vectorizador:


  alpha=0.01  -> F1-macro = 0.6730
  alpha=0.03  -> F1-macro = 0.6796
  alpha=0.05  -> F1-macro = 0.6836
  alpha=0.1   -> F1-macro = 0.6878
  alpha=0.3   -> F1-macro = 0.6930


  alpha=0.5   -> F1-macro = 0.6951
  alpha=1.0   -> F1-macro = 0.6921

>>> Mejor configuración: ComplementNB(alpha=0.5) con Tfidf afinado -> F1-macro = 0.6951


### Interpretación (consigna 3)

**Resultados obtenidos:**

| Vectorizador | Modelo | F1-macro |
|---|---|---|
| Count default | MultinomialNB | 0.5121 |
| Tfidf default | MultinomialNB | 0.5854 |
| Tfidf default | ComplementNB | 0.6930 |
| Tfidf + stop_words=en | ComplementNB | 0.6936 |
| Tfidf afinado (stop, min_df=2, max_df=0.95, sublinear_tf) | ComplementNB | 0.6921 |
| Tfidf afinado | MultinomialNB | 0.6437 |
| **Tfidf afinado** | **ComplementNB(alpha=0.5)** | **0.6951** ✅ |

**Lecturas:**

- **TF-IDF supera a CountVectorizer** (+0.07 F1 con el mismo modelo): pondera términos discriminativos y descuenta los muy frecuentes, atenuando el problema del supuesto de independencia de Naïve Bayes.
- **ComplementNB > MultinomialNB** (+0.11 F1 con el mismo vectorizador): es el cambio individual que más mejora. ComplementNB estima los parámetros usando el complemento de cada clase, lo que reduce el sesgo hacia clases mayoritarias en datasets con leve desbalance como 20 Newsgroups.
- **Filtros de vocabulario + sublinear_tf** dan una mejora pequeña en este caso. Aportarían más con menos datos o vocabularios más ruidosos.
- **Búsqueda de `alpha`**: el óptimo está cerca del default (alpha=0.5 dio 0.6951, alpha=1.0 dio 0.6921). Valores muy pequeños (0.01–0.05) empeoran el desempeño porque se sobreajusta a términos que aparecen pocas veces.
- El salto vs. el baseline de la consigna 2 (1-NN coseno = 0.5050) es enorme: **+0.19 F1-macro**, confirmando que el Naïve Bayes generativo aprovecha mucho mejor toda la información del train que un único vecino más cercano.

---
## **4. Similaridad entre palabras (matriz término-documento)**

Transponemos la matriz documento–término: ahora cada **fila** es un término representado por su distribución a lo largo de los documentos. La similaridad coseno entre dos filas mide cuán parecidamente se distribuyen dos palabras (aparecen en documentos similares ⇒ contextos similares).

Elegimos 5 palabras **manualmente** para que sean interpretables (no siglas raras ni typos).

In [8]:
# Reutilizamos un Tfidf con limpieza ligera para que el vocabulario sea más interpretable
vect_palabras = TfidfVectorizer(stop_words='english', min_df=5, max_df=0.5)
X_words = vect_palabras.fit_transform(newsgroups_train.data)
X_terms = X_words.T.tocsr()    # matriz término-documento: filas = términos

vocab = vect_palabras.vocabulary_
idx2word = {v: k for k, v in vocab.items()}
print(f'Vocabulario: {len(vocab)} términos | Matriz término-documento: {X_terms.shape}')

palabras_elegidas = ['car', 'god', 'computer', 'space', 'hockey']

for w in palabras_elegidas:
    if w not in vocab:
        print(f'"{w}" no está en el vocabulario, salteo.')
        continue
    i = vocab[w]
    sims = cosine_similarity(X_terms[i], X_terms)[0]
    top5 = np.argsort(sims)[::-1][1:6]    # excluyo la propia palabra
    print(f'\nPalabra: "{w}"  ->  top 5 más similares:')
    for j in top5:
        print(f'  {idx2word[j]:20s}  sim={sims[j]:.3f}')

Vocabulario: 17797 términos | Matriz término-documento: (17797, 11314)

Palabra: "car"  ->  top 5 más similares:
  cars                  sim=0.192
  dealer                sim=0.177
  civic                 sim=0.163
  loan                  sim=0.156
  owner                 sim=0.148

Palabra: "god"  ->  top 5 más similares:
  jesus                 sim=0.277
  bible                 sim=0.268
  christ                sim=0.267
  faith                 sim=0.255
  existence             sim=0.249

Palabra: "computer"  ->  top 5 más similares:
  shopper               sim=0.135
  verlag                sim=0.125
  delicate              sim=0.120
  drive                 sim=0.111
  hackers               sim=0.108

Palabra: "space"  ->  top 5 más similares:
  nasa                  sim=0.318
  shuttle               sim=0.278
  exploration           sim=0.233
  aeronautics           sim=0.222
  cfa                   sim=0.216

Palabra: "hockey"  ->  top 5 más similares:
  ncaa                  sim=0

### Interpretación (consigna 4)

**Resultados obtenidos:**

| Palabra | Top 5 más similares |
|---|---|
| `car` | cars, dealer, civic, loan, owner |
| `god` | jesus, bible, christ, faith, existence |
| `computer` | shopper, verlag, delicate, drive, hackers |
| `space` | nasa, shuttle, exploration, aeronautics, cfa |
| `hockey` | ncaa, nhl, sportschannel, players, league |

**Lecturas:**

- Para palabras temáticas muy específicas (`god`, `space`, `hockey`) los vecinos son **excelentes y muy interpretables**: `god → jesus, bible, christ, faith` es prácticamente un mini-tesauro religioso; `space → nasa, shuttle, exploration` lo mismo para astronáutica.
- Para `car` también aparecen vecinos del dominio automotor (cars, dealer, civic, owner), aunque con valores de similaridad más bajos porque el tema está más disperso en el corpus.
- **`computer` es el caso más débil**: aparecen términos genéricos (`shopper`, `delicate`, `drive`) o ruidosos (`verlag` viene de "Springer-Verlag" en posts de bibliografía, `hackers` solo aparece en pocos documentos pero co-ocurre fuertemente). Esto sucede porque "computer" es una palabra **demasiado frecuente y diseminada** por todo el corpus, así que su vector término-documento es poco discriminativo.
- Esto ilustra la **hipótesis distribucional** ("palabras en contextos similares tienen significados similares"): aunque TF-IDF es lineal y disperso, ya captura asociaciones temáticas razonables sin necesidad de embeddings densos.
- Limitaciones vs. Word2Vec/GloVe/fastText: esta similaridad es **temática** (agrupa por tema), no captura analogías ni sinonimia fina. Para `car` no aparecen sinónimos como `vehicle` o `automobile` salvo que co-ocurran fuerte en el mismo conjunto de documentos.
- Los filtros `min_df=5` y `max_df=0.5` son críticos: sin ellos, los vecinos serían typos, nombres propios o palabras casi universales sin contenido informativo.

---
## Conclusiones

| Método | F1-macro test |
|---|---|
| 1-NN coseno (prototipos) | 0.5050 |
| Count + MultinomialNB | 0.5121 |
| Tfidf + MultinomialNB | 0.5854 |
| Tfidf + ComplementNB | 0.6930 |
| **Tfidf afinado + ComplementNB(α=0.5)** | **0.6951** |

1. **Similaridad coseno + TF-IDF** captura bien la temática: ~70-80% de los vecinos comparten clase, y los "errores" caen en clases temáticamente vecinas.
2. **1-NN coseno** (0.5050) es un baseline razonable pero limitado: decide con un solo vecino y no agrega evidencia.
3. **El cambio más impactante** para mejorar F1 es pasar de `MultinomialNB` a `ComplementNB` (+0.11), seguido por usar TF-IDF en lugar de Count (+0.07). El tuning fino de `alpha` y filtros agrega ~+0.002.
4. **Transponer la matriz** da vectorizaciones de palabras útiles para palabras temáticas (`god`, `space`, `hockey`) pero falla con palabras muy frecuentes y diseminadas (`computer`), anticipando la motivación de los embeddings densos.